<a href="https://colab.research.google.com/github/binsue0/.github/blob/main/DL_day3/3_1_backprop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 딥러닝 구조 — 역전파로 좋은 가중치 찾기 

**핵심 질문:** 데이터에 가장 잘 맞는 가중치 `w`를 어떻게 찾을까?

- **순전파(forward):** 현재 `w`로 예측 → 손실(cost) 계산
- **역전파(backprop):** 손실의 기울기(gradient)를 구해 `w`를 개선 방향으로 갱신

오늘은 (1) 역전파를 **직접 손으로 구현**하고, (2) 같은 걸 PyTorch가 **자동으로** 해주는 걸 비교합니다.

## 1. 선형회귀 + 역전파 직접 구현

목표: `y = w·x` 형태로 데이터 `(1,2),(2,4),...,(5,10)`를 학습 → `w`가 2에 수렴해야 함.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

OSError: [WinError 1114] DLL 초기화 루틴을 실행할 수 없습니다. Error loading "c:\Users\sue71\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
x_data = torch.Tensor([1, 2, 3, 4, 5])
y_data = torch.Tensor([2, 4, 6, 8, 10])

x = x_data.view(5, 1)
y = y_data.view(5, 1) #reshape

print("x :", x)
print("y :", y)

가중치 `w`를 랜덤값으로 초기화하고, 예측 `w·x`를 확인합니다. 

In [ ]:
w = torch.rand(1,1)
w.item()

In [ ]:
w*x

### 학습 루프 
각 step마다:
1. **순전파:** `pre = w·x`
2. **손실:** `cost = mean((pre - y)²)`
3. **기울기(직접 미분):** `(w·x - y)²` 을 `w`로 미분하면 `2·(w·x - y)·x` 의 평균
4. **갱신:** `w ← w - lr·grad`

→ 직선이 점점 데이터에 맞춰지고 cost가 줄어드는 걸 그래프로 확인하세요.

In [ ]:
lr = 0.01

for step in range(20):
    pre = w*x
    cost = ((pre - y) ** 2).sum() / len(x)
    #(wx-y)^2 미분 시 2(wx-y)*x
    grad = 2*(pre-y).view(5).dot(x.view(5))/len(x)
    w -= lr*grad

    if step % 5 == 0 :
        plt.scatter(x.numpy(), y.numpy())
        plt.plot(x.numpy(), pre.numpy(), 'r-')
        # w.size() = 1*1, grad.size() = 1
        plt.title('step %d : cost=%.4f, w=%.4f, grad=%.4f' % (step, cost.item(), w.item(), grad.item()), fontdict={'size':15})
        plt.show()


학습된 `w`로 새로운 값 `x=6` 예측 → 12에 가까우면 성공.

In [ ]:
x_new = torch.Tensor([6])
y_new = w*x_new
y_new.item()

## 2. 같은 걸 PyTorch에게 맡기기 (nn.Linear + Optimizer)

방금 손으로 계산한 미분·갱신을, PyTorch는 `cost.backward()` 와 `optimizer.step()` 한 줄로 처리합니다.

| 직접 구현 | PyTorch |
|---|---|
| `grad = 2*(pre-y)·x / n` | `cost.backward()` |
| `w -= lr*grad` | `optimizer.step()` |

In [ ]:
model = nn.Linear(1, 1, bias = False)
model.weight

In [ ]:
loss = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
for step in range(30):
    pre = model(x)#순전파
    cost = loss(pre, y)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    if step % 5 == 0:
        plt.scatter(x.numpy(), y.numpy())
        # grad를 가진 tensor는 numpy()를 바로 사용할 수 없음
        # RuntimeError: Can't call numpy() on Variable that requires grad.
        plt.plot(x.numpy(), pre.data.numpy(), 'b-')
        plt.title('step %d, cost=%.4f, w=%.4f, grad=%4.f'
                  % (step, cost.item() ,model.weight.item(), model.weight.grad.item()), fontdict={'size':15})
        plt.show()

## 정리

- 역전파 = **기울기를 구해 손실이 줄어드는 방향으로 가중치를 조금씩 이동**시키는 과정
- 직접 미분하든, `backward()`로 자동 미분하든 **원리는 동일**
- 실무에선 PyTorch의 자동미분(autograd)을 쓰지만, 내부에서 일어나는 일은 Part 1 그대로입니다.